# 08 - Hyperparameter search

Search with Optuna over the hyperparameters in the `search.space` section of `configs/default.yaml`, on
`training.representation` and `training.feature_source`, with the window placement and the network variant (time
masking) of the configuration; every other setting comes from the `training` and `model` sections.

- **Cross-validation** (`search.cross_validation_folds` = k): the records are divided into k folds grouped by event
  and stratified by label (seed `split.random_seed`); every trial trains k models, each validated on one fold and
  trained on the others, and is scored by the mean of their best validation losses. With k = 1 every trial trains one
  model on the training/validation split of notebook 03. With time masking, records without any recorded data are
  excluded from every fold (they are classified bad by rule).
- **Sampler**: TPE, which proposes each trial from the results of the earlier ones.
- **Criterion**: mean best validation loss over the folds (weighted BCE: rewards both separation and calibration).
- **Pruning**: a trial whose validation loss is worse than the median of the earlier trials at the same epoch of the
  same fold is stopped; the first `search.pruner.warmup_epochs` epochs of every fold are never pruned, and pruning
  starts after `search.pruner.startup_trials` trials.
- **Resumable**: the study is stored in `search.search_dir/<study name>/study.db`, with study name
  `<search.study_name>_<representation>_<window placement>[_masked][_cv<k>]`, and one folder per trial (one
  subfolder per fold, checkpoint at every epoch). Run the notebook again after an interruption: the interrupted trial
  is resumed from its last epoch (after `search.grace_period_s` seconds), finished folds are not trained again, and
  the search continues until `search.n_trials` trials are finished.
- **Predictions**: with `search.save_fold_predictions: true` the deterministic predictions on the validation records
  of every finished fold are saved in its folder; section 5 computes the MC dropout predictions of the best trial on
  all the records, each predicted by the model that did not see it.

With `search.enqueue_baseline: true` the first trial is the configuration of the `model` and `training` sections
(baseline), trained under the same conditions as the others, as a reference for the gain of the search.

All models start from the same seed, so trials differ only in their hyperparameters. The best configuration is then to
be retrained with 2-3 seeds (notebook 05) before comparing it with others.

## 0. Setup

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import optuna
import pandas as pd
from optuna.visualization import matplotlib as optuna_plots

from wqu.config import load_config
from wqu.data.splitting import FOLD_COLUMN
from wqu.evaluation.padding import metrics_by_group
from wqu.training.experiment import experiment_name, fold_partitions, load_experiment_records, records_with_folds
from wqu.training.search import (
    BASELINE_ATTRIBUTE,
    VALIDATION_LOSS_STD_ATTRIBUTE,
    enqueue_baseline_trial,
    finished_trials,
    load_study,
    make_objective,
    out_of_fold_predictions,
    run_search,
    study_dir,
    variant_search,
)
from wqu.training.trainer import select_device
from wqu.visualization import apply_plot_style, save_figure

warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

CONFIG_PATH = Path("..") / "configs" / "default.yaml"

config = load_config(CONFIG_PATH)
columns = config.columns
training = config.training
search = variant_search(config.search, experiment_name(config))
n_folds = search.cross_validation_folds
apply_plot_style()
device = select_device(training.device)
output_dir = study_dir(search)
output_dir.mkdir(parents=True, exist_ok=True)

experiment = load_experiment_records(config)
if config.model.time_masking:
    print(f"Time masking on: {len(experiment.excluded):,} records without data excluded")
records = records_with_folds(config, experiment.records, n_folds)
print(f"Study {search.study_name}: {training.representation}, {training.feature_source}, "
      f"{n_folds} fold(s); device {device}")

## 1. Folds

Records, events and bad rate of the validation records of every fold, with the size of its training set,
the bad-class weight of its training records and the normalization statistics of its training records (stored in the
study folder and reused by every trial and session). With k = 1 the only fold is the validation set of the split.

In [ ]:
folds = fold_partitions(experiment, records, n_folds, training.representation, columns.label, output_dir)
pd.DataFrame([{
    "fold": fold.number,
    "validation_records": len(fold.partition.validation_records),
    "validation_events": fold.partition.validation_records[columns.event_id].nunique(),
    "validation_bad_rate": fold.partition.validation_records[columns.label].mean(),
    "training_records": len(fold.partition.train_records),
    "bad_weight": fold.partition.bad_weight,
} for fold in folds]).set_index("fold").round(3)

## 2. Study

The study is created on the first run and loaded afterwards; finished trials are kept.

In [ ]:
study = load_study(search)
if enqueue_baseline_trial(study, search, config.model, training):
    print("Baseline configuration queued as the first trial.")
print(f"Finished trials: {finished_trials(study)} of {search.n_trials}")
pd.Series({
    "learning_rate": f"{search.learning_rate_range} (log)",
    "conv_dropout": search.conv_dropout_choices,
    "recurrent_dropout": search.recurrent_dropout_range,
    "head_dropout": search.head_dropout_range,
    "residual_dropout": search.residual_dropout_options,
    "conv_filters": search.conv_filters_choices,
    "recurrent_units": search.recurrent_units_choices,
}, name="search space").to_frame()

## 3. Search

Runs trials until `search.n_trials` are finished. Each trial trains its folds in order and prints their epochs; a
pruned trial stops with a message. It can be interrupted at any time and resumed by running the notebook again.

In [ ]:
objective = make_objective(search, config.model, training, experiment, folds, columns.label, device)
started = run_search(study, objective, search.n_trials)
print(f"Trials run in this session: {started}; finished trials: {finished_trials(study)}")

## 4. Results

Completed trials sorted by mean best validation loss, then pruned trials (whose value is only the last reported loss,
not comparable with the mean over the folds), with the
hyperparameters, the standard deviation of the best validation loss over the folds, mean AUC, ECE and epoch of the best
model, and the duration; with k > 1 also the metrics of every fold (`fold_<k>_...`). Trials interrupted and resumed
under a new number are not listed twice (only the finished copy). Differences between trials smaller than the standard
deviation over the folds, or than the difference between seeds (notebook 08a), are not meaningful.

In [ ]:
trials = study.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs"))
trials = trials[trials["state"].isin(["COMPLETE", "PRUNED"])]
trials["params_conv_filters"] = trials["params_conv_filters"].map(lambda index: list(search.conv_filters_choices[int(index)]))
# Completed trials first: the value of a pruned trial is its last reported loss, of a fold that may differ.
trials = trials.sort_values(["state", "value"]).rename(columns=lambda name: name.removeprefix("params_").removeprefix("user_attrs_"))
if BASELINE_ATTRIBUTE in trials:
    trials[BASELINE_ATTRIBUTE] = trials[BASELINE_ATTRIBUTE].fillna(False).astype(bool)
trials.to_csv(output_dir / "trials.csv", index=False)
trials.round(4)

In [ ]:
best = study.best_trial
best_settings = {**best.params, "conv_filters": list(search.conv_filters_choices[best.params["conv_filters"]])}
if best_settings["residual_dropout"]:
    best_settings["residual_dropout"] = best_settings["conv_dropout"]
else:
    best_settings["residual_dropout"] = 0.0
(output_dir / "best_settings.json").write_text(json.dumps({"trial": best.number, "validation_loss": best.value,
                                                          **best_settings, **best.user_attrs}, indent=2))
print(f"Best trial: {best.number}, mean validation loss {best.value:.4f} "
      f"(standard deviation over the folds {best.user_attrs.get(VALIDATION_LOSS_STD_ATTRIBUTE, 0.0):.4f})")
baseline = [trial for trial in study.get_trials(deepcopy=False, states=(optuna.trial.TrialState.COMPLETE,))
            if trial.user_attrs.get(BASELINE_ATTRIBUTE)]
if baseline:
    print(f"Baseline (trial {baseline[0].number}): validation loss {baseline[0].value:.4f}; "
          f"gain of the search {baseline[0].value - best.value:.4f}")
print("Values for configs/default.yaml:")
print(f"  model.conv_filters: {best_settings['conv_filters']}")
print(f"  model.recurrent_units: {best_settings['recurrent_units']}")
for name in ("conv", "recurrent", "head", "residual"):
    print(f"  model.dropout.{name}: {best_settings[f'{name}_dropout']:.3f}")
print(f"  training.learning_rate: {best_settings['learning_rate']:.2e}")

## 5. Out-of-fold predictions of the best trial

MC dropout predictions (`model.mc_samples` passes) of the best trial: every record is predicted by the model of the fold
that held it out, so with k > 1 all the records get a prediction from a model that did not see them. The table of every
fold is cached in its folder; the joined table (same columns as the predictions of notebook 05, plus the fold) is saved
as `best_trial_predictions.csv`. Metrics per fold and on all the records: AUC and balanced accuracy on the raw
probabilities, ECE on the prior-corrected ones (each fold with the bad-class weight of its training records).

In [ ]:
best_predictions = out_of_fold_predictions(search, study.best_trial, config.model, training, experiment, folds,
                                           columns.label, device, config.model.mc_samples)
best_predictions.to_csv(output_dir / "best_trial_predictions.csv", index=False)
fold_groups = pd.Series(pd.Categorical(best_predictions[FOLD_COLUMN]))
everything = pd.Series(pd.Categorical(["all"] * len(best_predictions)))
pd.concat([
    metrics_by_group(best_predictions[probability].to_numpy(), best_predictions[f"{probability}_corrected"].to_numpy(),
                     best_predictions[columns.label].to_numpy(), groups, training.calibration_bins)
    for probability, groups in (("probability_mean", fold_groups), ("probability_mean", everything))
]).round(4)

## 6. Figures

Mean best validation loss per trial and best value so far; validation loss per reported step of every trial (step
`(fold - 1) * training.max_epochs + epoch`, so the folds follow each other; pruned trials end early); importance of each
hyperparameter for the validation loss (fANOVA, needs a few completed trials).

In [ ]:
FIGURES = {
    "optimization_history": optuna_plots.plot_optimization_history,
    "validation_loss_by_epoch": optuna_plots.plot_intermediate_values,
    "hyperparameter_importance": optuna_plots.plot_param_importances,
}
completed = len(study.get_trials(deepcopy=False, states=(optuna.trial.TrialState.COMPLETE,)))
for name, plot in FIGURES.items():
    if name == "hyperparameter_importance" and completed < 3:
        print("Hyperparameter importance needs at least 3 completed trials.")
        continue
    ax = plot(study)
    ax.figure.set_size_inches(9, 5)
    ax.figure.tight_layout()
    save_figure(ax.figure, output_dir / f"{name}.png")
    save_figure(ax.figure, output_dir / f"{name}.pdf")